# 스테이블코인 준비금 공시의 시장반응 분석
## 분석 코드 제출본 | stablecoin_master_v7.ipynb
---

| 항목 | 내용 |
|---|---|
| **논문 제목** | 스테이블코인 준비금 공시의 시장반응 분석 |
| **분석 대상** | USDT(19건) · USDC(8건) — 총 27개 공시 이벤트 (2021–2024) |
| **주 분석 창** | CAR(−5,+5) — 잔차 정규성 유일 통과 (SW p=0.625) |
| **시장 지수** | 비트코인(BTC) 일별 로그수익률 |
| **실행 환경** | Google Colab (Python 3.x) |

---

### ▶ 실행 순서

STEP 0 → STEP 1 → … → STEP 13 순서대로 **순차 실행**합니다.  
STEP 0의 Google Drive 마운트를 먼저 완료한 뒤 나머지 셀을 실행하세요.


---


---
## STEP 0 | 환경 설정

> 라이브러리 설치·임포트, 전역 파라미터(EST_WIN·EVT_WIN·PRIMARY_WIN 등), HAC maxlags 동적 함수, 오염 이벤트 제거 목록 정의.


In [ ]:
# ▼ Google Colab 실행 시 아래 두 줄 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/stablecoin_research'
DIRS = {k: f'{BASE}/{k}' for k in
        ['raw','events','processed','results','tables','figures','validation']}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

!pip install yfinance statsmodels scipy -q

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import (shapiro, jarque_bera, mannwhitneyu,
                         ttest_1samp, ttest_ind)
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
import yfinance as yf
warnings.filterwarnings('ignore')

!apt-get install -y fonts-nanum -qq > /dev/null
import matplotlib.font_manager as fm
fm.fontManager.ttflist.insert(0,
    fm.FontEntry(fname='/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
                 name='NanumGothic'))
plt.rcParams.update({'font.family':'NanumGothic','axes.unicode_minus':False,
                     'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})

# ── 전역 파라미터
START_COLLECT = '2020-07-01'
END_DATE      = '2025-04-30'
EST_WIN       = (-250, -11)
EVT_WIN       = (-10,  +10)
MIN_OBS       = 50
N_BOOT        = 5000
SEED          = 42
PRIMARY_WIN   = 'CAR_-5_5'
ALL_WINS      = ['CAR_-1_1', 'CAR_-3_3', 'CAR_-5_5']
np.random.seed(SEED)

# ──  HAC maxlags 동적 계산 함수
def get_maxlags(n):
    """소표본 수치 불안정 방지: N//3 기반 동적 제한, 최소 1 최대 5"""
    return min(5, max(1, int(n) // 3))

# ── 제거된 오염 이벤트 (분석 미포함, 투명성 기록용)
EXCLUDED_EVENTS = [
    {'coin':'USDT','date':'2021-10-07','reason':'2021-09-30과 이벤트창 14일 겹침'},
    {'coin':'USDC','date':'2023-03-13','reason':'2023-03-10(SVB)과 이벤트창 18일 겹침'},
]
pd.DataFrame(EXCLUDED_EVENTS).to_csv(
    f"{DIRS['validation']}/excluded_events.csv", index=False, encoding='utf-8-sig')

print('✅ 환경 설정 완료')
print(f'   주 분석 창: {PRIMARY_WIN}')
print(f'   HAC maxlags: 동적 (N//3, 최소 1 최대 5)')
print(f'   제거 이벤트: {[(e["coin"],e["date"]) for e in EXCLUDED_EVENTS]}')


---
## STEP 1 | 데이터 수집

> yfinance로 USDT·USDC·BTC 일별 가격·거래량 수집. 로그수익률, 패그이탈(|price−1|), 거래량 변화율 계산.


In [ ]:
TICKERS = {'USDT-USD':'USDT', 'USDC-USD':'USDC', 'BTC-USD':'BTC'}
raw_dfs = {}

for ticker, sym in TICKERS.items():
    print(f'🔄 {sym}...')
    df = yf.download(ticker, start=START_COLLECT, end=END_DATE,
                     interval='1d', auto_adjust=True, progress=False)
    df = df.reset_index()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([c for c in col if c]).strip('_') for col in df.columns]
    df.columns = [c.lower() for c in df.columns]
    close_col  = next(c for c in df.columns if 'close'  in c)
    volume_col = next(c for c in df.columns if 'volume' in c)
    date_col   = next(c for c in df.columns if 'date'   in c)
    df = df.rename(columns={date_col:'date', close_col:'price', volume_col:'volume'})
    df = df[['date','price','volume']].copy()
    df['date']       = pd.to_datetime(df['date']).dt.normalize()
    df               = df.dropna(subset=['price']).sort_values('date').reset_index(drop=True)
    df['coin']       = sym
    df['log_return'] = np.log(df['price'] / df['price'].shift(1))
    df['price_dev']  = (df['price'] - 1.0).abs()
    df['vol_chg']    = np.log(df['volume'] / df['volume'].shift(1))
    df.to_csv(f"{DIRS['raw']}/{sym}.csv", index=False)
    raw_dfs[sym] = df
    print(f'   ✅ {sym}: {len(df)}일  ({df.date.min().date()} ~ {df.date.max().date()})')
    time.sleep(1)


---
## STEP 2 | 이벤트 정의 (27건)

> 공시 품질지수 함수 `score()` 정의 (1~5점). 27건 이벤트 목록 등록. 오염 이벤트 2건(Bloomberg 2021-10-07, 패그회복 2023-03-13) 별도 제거.


In [ ]:
def score(amt=True, composition=False, detail=False, audit=False):
    """공시 품질 지수 1~5점 (인자명 명확화)
    ①amt: 준비금 총액 공시  ②composition: 자산 구성 공시
    ③detail: 세부 발행사/만기  ④audit: 제3자 독립 감사
    반환: 충족 기준 수 + 1  →  1점(모두 미충족) ~ 5점(모두 충족)
    """
    return int(amt) + int(composition) + int(detail) + int(audit) + 1

# ── 최종 이벤트 목록 (오염 이벤트 제거 후)
# 제거: USDT 2021-10-07 (Bloomberg) — 2021-09-30과 창 14일 겹침
# 제거: USDC 2023-03-13 (패그회복)  — 2023-03-10(SVB)과 창 18일 겹침
events_raw = [
    # ── USDT
    {'coin':'USDT','event_date':'2021-03-31','event_type':'neutral',
     'quality_score':score(True,False,False,False),
     'description':'Q1 2021 증명보고서 — 준비금 총액만 공개',
     'source':'Tether Transparency Page',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2021-05-13','event_type':'negative',
     'quality_score':score(False,False,False,False),
     'description':'CFTC 4,100만달러 벌금 — 준비금 허위표시 합의',
     'source':'CFTC Press Release No. 8450-21',
     'source_url':'https://www.cftc.gov/PressRoom/PressReleases/8450-21'},
    {'coin':'USDT','event_date':'2021-06-30','event_type':'neutral',
     'quality_score':score(True,False,False,False),
     'description':'Q2 2021 증명보고서',
     'source':'Tether Transparency Page',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2021-08-20','event_type':'positive',
     'quality_score':score(True,True,False,False),
     'description':'준비금 구성 첫 공개 — 상업어음 49%, 미국채 10% 등',
     'source':'Tether Transparency Report (Q2 2021 breakdown)',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2021-09-30','event_type':'neutral',
     'quality_score':score(True,True,False,False),
     'description':'Q3 2021 증명보고서 — 자산군 구분 공개',
     'source':'Tether Transparency Page',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2021-12-31','event_type':'neutral',
     'quality_score':score(True,True,False,False),
     'description':'Q4 2021 증명보고서',
     'source':'Tether Transparency Page',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2022-03-31','event_type':'positive',
     'quality_score':score(True,True,True,False),
     'description':'Q1 2022 — 상업어음 24%로 축소, 세부 발행사 공개',
     'source':'Tether Q1 2022 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2022-06-30','event_type':'positive',
     'quality_score':score(True,True,True,False),
     'description':'Q2 2022 — 미국채 비중 확대, 상업어음 8.6%로 감소',
     'source':'Tether Q2 2022 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2022-09-30','event_type':'neutral',
     'quality_score':score(True,True,True,False),
     'description':'Q3 2022 증명보고서',
     'source':'Tether Q3 2022 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2022-11-11','event_type':'negative',
     'quality_score':score(False,False,False,False),
     'description':'FTX 파산 신청일 — 시장 전반 패닉, USDT 일시 디페그',
     'source':'FTX Bankruptcy Filing (Chapter 11) 2022-11-11',
     'source_url':'https://cases.ra.kroll.com/FTX/'},
    {'coin':'USDT','event_date':'2022-12-31','event_type':'positive',
     'quality_score':score(True,True,True,False),
     'description':'Q4 2022 — 상업어음 0% 달성, 미국채 58.1% 전환',
     'source':'Tether Q4 2022 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2023-03-31','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q1 2023 — BDO Italia 제3자 감사, 미국채 100% 전환 완료',
     'source':'Tether Q1 2023 Attestation Report (BDO Italia)',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2023-06-30','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q2 2023 — BDO Italia 감사',
     'source':'Tether Q2 2023 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2023-09-30','event_type':'neutral',
     'quality_score':score(True,True,True,True),
     'description':'Q3 2023 증명보고서',
     'source':'Tether Q3 2023 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2023-12-31','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q4 2023 — 순이익 39억달러, 잉여 준비금 공시',
     'source':'Tether Q4 2023 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2024-03-31','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q1 2024 증명보고서',
     'source':'Tether Q1 2024 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2024-06-30','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q2 2024 증명보고서',
     'source':'Tether Q2 2024 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2024-09-30','event_type':'neutral',
     'quality_score':score(True,True,True,True),
     'description':'Q3 2024 증명보고서',
     'source':'Tether Q3 2024 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    {'coin':'USDT','event_date':'2024-12-31','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q4 2024 증명보고서',
     'source':'Tether Q4 2024 Attestation Report',
     'source_url':'https://tether.to/en/transparency'},
    # ── USDC
    {'coin':'USDC','event_date':'2021-01-31','event_type':'neutral',
     'quality_score':score(True,True,False,False),
     'description':'USDC 월별 증명보고서 (MIN_OBS 미달 → STEP 4 자동 제외)',
     'source':'Circle Transparency Page',
     'source_url':'https://www.circle.com/en/transparency'},
    {'coin':'USDC','event_date':'2021-07-08','event_type':'positive',
     'quality_score':score(True,True,True,False),
     'description':'Circle 공식 발표 — 현금+미국채만 보유 선언',
     'source':'Circle Blog 2021-07-08',
     'source_url':'https://www.circle.com/blog/usdc-reserve-assets'},
    {'coin':'USDC','event_date':'2022-01-20','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Grant Thornton 공인 감사보고서 최초 발행',
     'source':'Grant Thornton USDC Reserve Attestation January 2022',
     'source_url':'https://www.circle.com/en/transparency'},
    {'coin':'USDC','event_date':'2022-06-30','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Deloitte 감사법인 전환 — 준비금 상세 공시 강화',
     'source':'Circle Transparency Report Q2 2022',
     'source_url':'https://www.circle.com/en/transparency'},
    {'coin':'USDC','event_date':'2023-03-10','event_type':'negative',
     'quality_score':score(False,False,False,False),
     'description':'SVB 파산 — USDC 준비금 33억달러 SVB 예치 노출 공시',
     'source':'Circle Twitter @circle 2023-03-10; FDIC PR-16-2023',
     'source_url':'https://www.fdic.gov/news/press-releases/2023/pr23016.html'},
    {'coin':'USDC','event_date':'2023-06-30','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q2 2023 Deloitte 감사 증명보고서',
     'source':'Circle Transparency Report Q2 2023',
     'source_url':'https://www.circle.com/en/transparency'},
    {'coin':'USDC','event_date':'2023-12-31','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q4 2023 Deloitte 감사 증명보고서',
     'source':'Circle Transparency Report Q4 2023',
     'source_url':'https://www.circle.com/en/transparency'},
    {'coin':'USDC','event_date':'2024-03-31','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q1 2024 Deloitte 감사 증명보고서',
     'source':'Circle Transparency Report Q1 2024',
     'source_url':'https://www.circle.com/en/transparency'},
    {'coin':'USDC','event_date':'2024-09-30','event_type':'positive',
     'quality_score':score(True,True,True,True),
     'description':'Q3 2024 Deloitte 감사 증명보고서',
     'source':'Circle Transparency Report Q3 2024',
     'source_url':'https://www.circle.com/en/transparency'},
]

events_df = pd.DataFrame(events_raw)
events_df['event_date'] = pd.to_datetime(events_df['event_date'])
events_df = events_df.sort_values('event_date').reset_index(drop=True)
events_df.to_csv(f"{DIRS['events']}/events_verified.csv",
                 index=False, encoding='utf-8-sig')

print(f'✅ 이벤트: {len(events_df)}건 등록')
print(f'   USDT {(events_df.coin=="USDT").sum()}건 / USDC {(events_df.coin=="USDC").sum()}건')
print(f'   (USDC 2021-01-31 포함 — STEP 4 MIN_OBS 미달 시 자동 제외)')
print()
print(events_df.groupby(['coin','event_type']).size().rename('건수').to_string())


---
## STEP 3 | 데이터 전처리 & 기초통계

> USDT·USDC·BTC DataFrame inner join → master_daily (약 1,763일). 기초통계 산출.


In [ ]:
usdt = raw_dfs['USDT']; usdc = raw_dfs['USDC']; btc = raw_dfs['BTC']

master = (
    usdt[['date','price','volume','log_return','price_dev','vol_chg']]
    .rename(columns={'price':'p_usdt','volume':'v_usdt','log_return':'r_usdt',
                     'price_dev':'dev_usdt','vol_chg':'vc_usdt'})
    .merge(
        usdc[['date','price','volume','log_return','price_dev','vol_chg']]
        .rename(columns={'price':'p_usdc','volume':'v_usdc','log_return':'r_usdc',
                         'price_dev':'dev_usdc','vol_chg':'vc_usdc'}),
        on='date', how='inner')
    .merge(btc[['date','log_return']].rename(columns={'log_return':'r_btc'}),
           on='date', how='inner')
    .sort_values('date').reset_index(drop=True)
)
master['t'] = range(len(master))
master.to_csv(f"{DIRS['processed']}/master_daily.csv", index=False)

desc = master[['r_usdt','r_usdc','r_btc','dev_usdt','dev_usdc']].describe().round(8)
desc.to_csv(f"{DIRS['tables']}/Table1_descriptive.csv", encoding='utf-8-sig')

print(f'✅ 마스터: {len(master)}일  ({master.date.min().date()} ~ {master.date.max().date()})')
display(desc)


---
## STEP 4 | 시장모형 추정 & CAR 계산

> 핵심 계산 블록. `estimate_params()`: 추정창 OLS(HAC) 추정. `compute_car()`: 이벤트창 AR/CAR 계산. `make_event_level()`: 4개 창 CAR 집계.


In [ ]:
def estimate_params(master, event_date, ret_col, mkt_col='r_btc'):
    """시장모형 OLS(HAC) 추정. None 반환 시 분석 제외."""
    idx = master.index[master['date'] == event_date].tolist()
    if not idx:
        return None
    t0 = idx[0]
    s, e = t0 + EST_WIN[0], t0 + EST_WIN[1]
    if s < 0:
        return None
    sub = master.iloc[s:e+1].dropna(subset=[ret_col, mkt_col])
    if len(sub) < MIN_OBS:
        return None
    # HAC maxlags 동적 제한
    lag = get_maxlags(len(sub))
    X   = sm.add_constant(sub[mkt_col])
    res = sm.OLS(sub[ret_col], X).fit(cov_type='HAC', cov_kwds={'maxlags': lag})
    dw            = durbin_watson(res.resid)
    _, bp_p, _, _ = het_breuschpagan(res.resid, res.model.exog)
    return {
        'alpha': res.params['const'],
        'beta':  res.params[mkt_col],
        'r2':    res.rsquared,
        'sigma': res.resid.std(ddof=2),
        't0':    t0,
        'dw':    dw,
        'bp_p':  bp_p,
        'n_est': len(sub),
        'lag':   lag,
    }


def compute_car(master, events_df, coin, ret_col):
    """이벤트별 CAR(tau 단위) 계산."""
    car_rows, skipped, diag_rows = [], [], []

    for _, ev in events_df[events_df['coin'] == coin].iterrows():
        params = estimate_params(master, ev['event_date'], ret_col)
        if params is None:
            skipped.append({'coin': coin,
                            'event_date': str(ev['event_date'].date()),
                            'reason': 'MIN_OBS 미달 또는 날짜 없음'})
            continue

        diag_rows.append({
            'event_date': str(ev['event_date'].date()), 'coin': coin,
            'DW':       round(params['dw'],   6),
            'DW_판정':  '✅' if 1.5 <= params['dw'] <= 2.5 else '⚠',
            'BP_p':     round(params['bp_p'], 6),
            'BP_판정':  '✅' if params['bp_p'] > 0.05 else '⚠',
            'r2':       round(params['r2'],   6),
            'n_est':    params['n_est'],
            'lag_used': params['lag'],
        })

        t0 = params['t0']
        ws = max(0, t0 + EVT_WIN[0])
        we = min(len(master) - 1, t0 + EVT_WIN[1])
        win = master.iloc[ws:we+1].copy().dropna(subset=[ret_col, 'r_btc'])
        win['tau']   = win.index - t0
        win['E_ret'] = params['alpha'] + params['beta'] * win['r_btc']
        win['AR']    = win[ret_col] - win['E_ret']
        win['SAR']   = win['AR'] / params['sigma'] if params['sigma'] > 0 else np.nan
        #  win['CAR']은 τ=-10부터 시작하는 누적 AR (시각화용)
        #  CAR(lo,hi)는 make_event_level()의 AR.sum()으로 별도 계산
        win['CAR']   = win['AR'].cumsum()
        win['coin']  = coin
        for col in ['event_date', 'event_type', 'quality_score', 'description']:
            win[col] = ev[col]
        win[['alpha','beta','r2','sigma']] = (
            params['alpha'], params['beta'], params['r2'], params['sigma'])
        car_rows.append(win)

    if skipped:
        pd.DataFrame(skipped).to_csv(
            f"{DIRS['validation']}/{coin}_skipped.csv", index=False)
        print(f'  ⚠ {coin} 추정 불가: {[s["event_date"] for s in skipped]}')

    car_df  = pd.concat(car_rows, ignore_index=True) if car_rows else pd.DataFrame()
    diag_df = pd.DataFrame(diag_rows)
    return car_df, diag_df


def make_event_level(car_df, wins=[(-1,1),(-3,3),(-5,5),(-10,10)]):
    rows = []
    for ed, grp in car_df.groupby('event_date'):
        row = {
            'event_date':    ed,
            'coin':          grp['coin'].iloc[0],
            'event_type':    grp['event_type'].iloc[0],
            'quality_score': grp['quality_score'].iloc[0],
            'description':   grp['description'].iloc[0],
        }
        for lo, hi in wins:
            sub = grp[grp['tau'].between(lo, hi)]
            row[f'CAR_{lo}_{hi}'] = sub['AR'].sum() if len(sub) > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


print('🔄 USDT CAR 계산...')
car_usdt, diag_usdt = compute_car(master, events_df, 'USDT', 'r_usdt')
print('🔄 USDC CAR 계산...')
car_usdc, diag_usdc = compute_car(master, events_df, 'USDC', 'r_usdc')

ev_usdt = make_event_level(car_usdt)
ev_usdc = make_event_level(car_usdc)
ev_all  = pd.concat([ev_usdt, ev_usdc], ignore_index=True)
ev_all['negative'] = (ev_all['event_type'] == 'negative').astype(int)
ev_all['is_usdc']  = (ev_all['coin'] == 'USDC').astype(int)

diag_all = (pd.concat([diag_usdt, diag_usdc])
            .sort_values(['coin','event_date']).reset_index(drop=True))
diag_all.to_csv(f"{DIRS['validation']}/market_model_diagnostics.csv",
                index=False, encoding='utf-8-sig')
car_usdt.to_csv(f"{DIRS['processed']}/car_usdt.csv", index=False)
car_usdc.to_csv(f"{DIRS['processed']}/car_usdc.csv", index=False)
ev_all.to_csv(f"{DIRS['processed']}/event_level_car.csv", index=False)

dw_w = (diag_all['DW_판정'] == '⚠').sum()
bp_w = (diag_all['BP_판정'] == '⚠').sum()
print(f'\n✅ USDT {len(ev_usdt)}건 / USDC {len(ev_usdc)}건 (합계 {len(ev_all)}건)')
print(f'📋 시장모형 진단: DW ⚠ {dw_w}건 / BP ⚠ {bp_w}건')
print(f'   HAC maxlags: {diag_all["lag_used"].describe().to_dict()}')


---
## STEP 5 | τ별 평균 CAR 요약

> `tau_summary()`: τ별 mean_AR 집계 → mean_CAR = cumsum(mean_AR) (이론 정합).


In [ ]:
def tau_summary(car_df, label=''):
    """tau별 평균 AR 집계 및 검정.
      - mean_CAR: groupby 집계 → mean_AR.cumsum() (이론적 정합성)
      - p_value df: 고정 iloc[0] → 행별 실제 n (df 버그 수정)
    """
    s = (car_df.groupby('tau')
         .agg(mean_AR=('AR','mean'), std_AR=('AR','std'), n=('AR','count'))
         .reset_index())
    s['se']      = s['std_AR'] / np.sqrt(s['n'])
    s['t_stat']  = s['mean_AR'] / s['se'].replace(0, np.nan)
    #  df를 행별 n으로 계산 (고정 iloc[0] 버그 수정)
    s['p_value'] = s.apply(
        lambda row: (2*(1 - stats.t.cdf(abs(row['t_stat']), df=row['n']-1))
                     if pd.notna(row['t_stat']) and row['n'] > 1 else np.nan),
        axis=1)
    #  mean_CAR = mean_AR 누적합 (이벤트창 기준 올바른 정의)
    s = s.sort_values('tau').reset_index(drop=True)
    s['mean_CAR'] = s['mean_AR'].cumsum()
    if label:
        s['coin'] = label
    return s

sum_usdt = tau_summary(car_usdt, 'USDT')
sum_usdc = tau_summary(car_usdc, 'USDC')
car_summary = pd.concat([sum_usdt, sum_usdc], ignore_index=True)
car_summary.to_csv(f"{DIRS['processed']}/car_summary_by_tau.csv", index=False)
print(f'✅ USDT N={sum_usdt["n"].mode()[0]}  USDC N={sum_usdc["n"].mode()[0]}')


---
## STEP 6 | H1 검정 — CAR = 0

> `bootstrap_ci()` 5,000회 복원추출. t검정 + Bootstrap 95% CI 병행. 불일치 자동 감지.


In [ ]:
def bootstrap_ci(arr, n=N_BOOT, alpha=0.05):
    arr   = np.array(arr)
    boots = [np.random.choice(arr, len(arr), replace=True).mean()
             for _ in range(n)]
    return np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])

print('='*70)
print('📌 H1: CAR = 0  (t-test + Bootstrap 95% CI)')
print('='*70)
h1_rows = []

# SVB 이벤트 날짜 (USDC 평균 주석용)
SVB_DATE = '2023-03-10'

for car_col in ALL_WINS + ['CAR_-10_10']:
    for coin in ['USDT', 'USDC']:
        vals = ev_all[(ev_all['coin']==coin) & ev_all[car_col].notna()][car_col].values
        if len(vals) < 2:
            continue
        t_stat, p = ttest_1samp(vals, 0)
        lo, hi    = bootstrap_ci(vals)
        sig       = ('***' if p<0.01 else '**' if p<0.05
                     else '*' if p<0.1 else 'n.s.')
        primary   = ' ★' if car_col == PRIMARY_WIN else ''

        #  Bootstrap CI vs t-test 불일치 경고
        ci_excl0 = (lo > 0 or hi < 0)
        conflict = ci_excl0 and p >= 0.05 or (not ci_excl0 and p < 0.05)
        cnfl_note = '  ⚠ CI·t-test 불일치' if conflict else ''

        print(f'{coin} {car_col:14s}  N={len(vals):2d}  '
              f'mean={vals.mean()*10000:+8.3f}bp  '
              f't={t_stat:6.3f}  p={p:.4f} {sig:5s}  '
              f'Boot95%=[{lo*10000:+.3f}, {hi*10000:+.3f}]{primary}{cnfl_note}')

        # [v5 추가] USDC: SVB 이벤트 제외 평균 병기
        svb_note = ''
        if coin == 'USDC':
            ev_excl = ev_all[(ev_all['coin']=='USDC') &
                              ev_all[car_col].notna() &
                              (ev_all['event_date'].dt.strftime('%Y-%m-%d') != SVB_DATE)]
            if len(ev_excl) > 0:
                m_excl = ev_excl[car_col].mean()*10000
                svb_note = (f'   └ SVB 제외 mean={m_excl:+.3f}bp (N={len(ev_excl)})'
                            f' — SVB 이벤트 1건이 평균을 {(vals.mean()-ev_excl[car_col].mean())*10000:+.2f}bp 이동')
                print(svb_note)

        # coin별 분기 처리로 USDT가 USDC 데이터 재참조하던 잠재버그 수정
        if coin == 'USDC':
            _excl = ev_all[
                (ev_all['coin']=='USDC') &
                ev_all[car_col].notna() &
                (ev_all['event_date'].dt.strftime('%Y-%m-%d') != SVB_DATE)]
            mean_excl_svb = _excl[car_col].mean() * 10000 if len(_excl) > 0 else None
        else:
            mean_excl_svb = None  # USDT: SVB 해당 없음

        h1_rows.append({
            'coin': coin, 'window': car_col, 'N': len(vals),
            'mean_bp': vals.mean()*10000,
            'mean_bp_excl_svb': mean_excl_svb,
            't': t_stat, 'p': p,
            'boot_lo_bp': lo*10000, 'boot_hi_bp': hi*10000,
            'ci_excludes_zero': ci_excl0,
            'conflict': bool(conflict),
            'primary': car_col == PRIMARY_WIN,
        })

pd.DataFrame(h1_rows).to_csv(f"{DIRS['results']}/H1_results.csv", index=False)
print('\n✅ H1 결과 저장  (mean_bp_excl_svb 컬럼 추가)')


---
## STEP 7 | H2 검정 — 공시품질 × CAR

> `reg_with_diag()`: OLS 횡단면 회귀 + 잔차 진단 5종(DW·BP·JB·SW·VIF). 주 창: CAR(−5,+5). SE 유형 자동 선택(HC3 or HAC).


In [ ]:
def reg_with_diag(y, X_df, label=''):
    """OLS 회귀 + 잔차 진단 +  VIF 절편 제외."""
    X      = sm.add_constant(X_df)
    n      = len(y)
    # 
    #  HAC maxlags 동적
    lag    = get_maxlags(n)
    m_hc3  = sm.OLS(y, X).fit(cov_type='HC3')
    m_hac  = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lag})
    dw     = durbin_watson(m_hc3.resid)
    _, bp_p, _, _ = het_breuschpagan(m_hc3.resid, m_hc3.model.exog)
    jb_s, jb_p    = jarque_bera(m_hc3.resid)
    _, sw_p       = shapiro(m_hc3.resid)
    #  VIF: 절편(인덱스 0) 제외 — 인덱스 1부터
    vif    = [variance_inflation_factor(X.values, i)
              for i in range(1, X.shape[1])]
    use_hac = not (1.5 <= dw <= 2.5)
    m_use   = m_hac if use_hac else m_hc3
    diag = {
        'DW': dw, 'DW_ok': 1.5<=dw<=2.5,
        'BP_p': bp_p, 'JB_p': jb_p, 'SW_p': sw_p,
        'VIF_max': max(vif) if vif else np.nan,
        'VIF_vars': dict(zip(X_df.columns, [round(v,3) for v in vif])),
        'R2': m_hc3.rsquared, 'AdjR2': m_hc3.rsquared_adj,
        'N': n, 'lag': lag,
        'SE': 'HAC' if use_hac else 'HC3',
    }
    return m_use, diag


print('='*70)
print(f'📌 H2: quality_score → CAR  |  주 창: {PRIMARY_WIN}')
print('='*70)
h2_rows, diag_log, reg_models = [], [], {}

for car_col in ALL_WINS:
    sub   = ev_all.dropna(subset=[car_col])
    xcols = ['quality_score', 'negative', 'is_usdc']
    m, diag = reg_with_diag(sub[car_col], sub[xcols])
    reg_models[car_col] = (m, diag)

    primary  = '  ★ 주 분석 창' if car_col == PRIMARY_WIN else ''
    norm_f   = '✅' if diag['SW_p'] > 0.05 else '❌'
    dw_f     = '✅' if diag['DW_ok'] else f'⚠ ({diag["DW"]:.3f})'
    bp_f     = '✅' if diag['BP_p'] > 0.05 else '⚠'
    vif_f    = '✅' if diag['VIF_max'] < 5 else '⚠'

    print(f'\n[{car_col}]{primary}  N={diag["N"]}  '
          f'R²={diag["R2"]:.4f}  Adj-R²={diag["AdjR2"]:.4f}  '
          f'SE={diag["SE"]}(lag={diag["lag"]})')

    #  Adj-R² 음수 경고
    if diag['AdjR2'] < 0:
        print(f'  ⚠ Adj-R²={diag["AdjR2"]:.4f} 음수 — null 모형보다 적합 불량, 해석 주의')

    print(f'  진단 → DW={dw_f}  SW_p={diag["SW_p"]:.4f}{norm_f}  '
          f'JB_p={diag["JB_p"]:.4e}  '
          f'BP_p={diag["BP_p"]:.4f}{bp_f}  '
          f'VIF_max={diag["VIF_max"]:.3f}{vif_f}')
    print(f'  VIF 상세: {diag["VIF_vars"]}')

    for var in ['const', 'quality_score', 'negative', 'is_usdc']:
        coef = m.params[var]
        t_   = m.tvalues[var]
        p_   = m.pvalues[var]
        sig  = '***' if p_<0.01 else '**' if p_<0.05 else '*' if p_<0.1 else 'n.s.'
        print(f'  {var:20s}  {coef*10000:+10.4f}bp  t={t_:6.3f}  p={p_:.4f} {sig}')
        h2_rows.append({
            'window': car_col, 'var': var,
            'coef': coef, 'coef_bp': coef*10000,
            't': t_, 'p': p_, 'r2': diag['R2'], 'adj_r2': diag['AdjR2'],
            'se_type': diag['SE'], 'lag': diag['lag'],
            'primary': car_col == PRIMARY_WIN,
        })
    diag_log.append({'window': car_col,
                     **{k: v for k,v in diag.items()
                        if k not in ['DW_ok','VIF_vars']}})

h2_df   = pd.DataFrame(h2_rows)
diag_df = pd.DataFrame(diag_log)
h2_df.to_csv(f"{DIRS['results']}/H2_regression.csv",             index=False)
h2_df.to_csv(f"{DIRS['tables']}/Table3_H2_regression.csv",       index=False, encoding='utf-8-sig')
diag_df.to_csv(f"{DIRS['results']}/H2_residual_diagnostics.csv", index=False)

with open(f"{DIRS['results']}/H2_full_output.txt", 'w', encoding='utf-8') as f:
    for col, (m, diag) in reg_models.items():
        f.write(f'\n===== {col} (SE={diag["SE"]}, lag={diag["lag"]}) =====\n'
                f'{m.summary().as_text()}\n')

print('\n✅ H2 결과 저장')


---
## STEP 8 | H3 검정 — 긍정 vs 부정 비대칭성

> `calc_ratio()`, `describe_direction()`. Welch t + Mann-Whitney U 검정. USDT 도피처 효과 분석.


In [ ]:
def calc_ratio(pos_m, neg_m):
    """동일 부호이면 ratio=None (의미 없는 비율 방지)."""
    if pd.isna(pos_m) or pd.isna(neg_m):
        return None, 'N/A'
    if abs(pos_m) < 1e-9:
        return None, '분모≈0: 비율 산출 불가'
    # [v5.1 수정] 같은 부호이면 ratio=None 반환 (절대값 비율 저장 안 함)
    if (pos_m > 0 and neg_m > 0):
        return None, '같은 부호(도피처효과) — 비율 산출 불가'
    if (pos_m < 0 and neg_m < 0):
        return None, '같은 부호(둘 다 음수) — 비율 산출 불가'
    # 방향 반대: 정상 비대칭 비율
    return abs(neg_m / pos_m), 'H3 방향 일치'


def describe_direction(pos_m, neg_m):
    if pd.isna(pos_m) or pd.isna(neg_m):
        return 'N/A'
    if neg_m < 0 <= pos_m:
        return '음수 — H3 방향 일치'
    if neg_m > 0 > pos_m:
        return '역전 (긍정이 음수)'
    if neg_m > 0 and pos_m > 0:
        return ('양수>긍정 — 도피처효과'
                if neg_m > pos_m else '양수<긍정 — 비대칭 미약')
    if neg_m < 0 and pos_m < 0:
        return '둘 다 음수 — 비율 무의미'
    return f'neg={neg_m:.3f} pos={pos_m:.3f}'


print('='*70)
print('📌 H3: 긍정 vs 부정 비대칭성')
print(f'   주 창: {PRIMARY_WIN}')
print('='*70)
h3_rows = []

for coin in ['USDT', 'USDC']:
    for car_col in ALL_WINS:
        pos = ev_all[(ev_all['coin']==coin) &
                     (ev_all['event_type']=='positive')][car_col].dropna()
        neg = ev_all[(ev_all['coin']==coin) &
                     (ev_all['event_type']=='negative')][car_col].dropna()
        n_pos, n_neg = len(pos), len(neg)
        pos_m = pos.mean()*10000 if n_pos > 0 else np.nan
        neg_m = neg.mean()*10000 if n_neg > 0 else np.nan

        ratio, ratio_note = calc_ratio(pos_m, neg_m)
        direction         = describe_direction(pos_m, neg_m)
        primary           = ' ★' if car_col == PRIMARY_WIN else ''
        ratio_str = (f'{ratio:.2f}x' if ratio is not None
                     else f'N/A ({ratio_note})')

        if n_pos >= 2 and n_neg >= 2:
            _, p_t  = ttest_ind(pos, neg, equal_var=False)
            _, p_mw = mannwhitneyu(pos, neg, alternative='two-sided')
            sig_t   = ('***' if p_t<0.01 else '**' if p_t<0.05
                       else '*' if p_t<0.1 else 'n.s.')
            sig_mw  = ('***' if p_mw<0.01 else '**' if p_mw<0.05
                       else '*' if p_mw<0.1 else 'n.s.')
            mw_note = '  ⚠ N_neg=2: MW 검정력 매우 낮음' if n_neg == 2 else ''
            print(f'{coin} {car_col:12s}  '
                  f'긍정={pos_m:+8.3f}bp(N={n_pos})  '
                  f'부정={neg_m:+8.3f}bp(N={n_neg})  '
                  f'비율={ratio_str}  '
                  f't-p={p_t:.4f}{sig_t}  MW-p={p_mw:.4f}{sig_mw}{primary}{mw_note}')
            h3_rows.append({
                'coin': coin, 'window': car_col,
                'pos_mean_bp': pos_m, 'neg_mean_bp': neg_m,
                'ratio': ratio, 'ratio_note': ratio_note,
                'p_ttest': p_t, 'p_mw': p_mw,
                'n_pos': n_pos, 'n_neg': n_neg,
                'test_feasible': True, 'direction': direction,
                'primary': car_col == PRIMARY_WIN,
            })
        else:
            print(f'{coin} {car_col:12s}  '
                  f'긍정={pos_m:+8.3f}bp(N={n_pos})  '
                  f'부정={neg_m:+8.3f}bp(N={n_neg})  '
                  f'비율={ratio_str}  '
                  f'⚠ N_neg={n_neg} — 검정 불가, 서술 한정{primary}')
            h3_rows.append({
                'coin': coin, 'window': car_col,
                'pos_mean_bp': pos_m, 'neg_mean_bp': neg_m,
                'ratio': ratio, 'ratio_note': ratio_note,
                'p_ttest': None, 'p_mw': None,
                'n_pos': n_pos, 'n_neg': n_neg,
                'test_feasible': False, 'direction': direction,
                'primary': car_col == PRIMARY_WIN,
            })

h3_df = pd.DataFrame(h3_rows)
h3_df.to_csv(f"{DIRS['results']}/H3_results.csv",         index=False)
h3_df.to_csv(f"{DIRS['tables']}/Table4_H3_asymmetry.csv", index=False, encoding='utf-8-sig')
print('\n⚠ USDC 부정 N=1 — 통계검정 불가, 사례 서술 한정')
print('⚠ USDT 부정 N=2 — MW 검정력 매우 낮음')
print('✅ ratio: 동일 부호이면 None 저장 (Table4에서 N/A로 표시됨)')
print('\n✅ H3 결과 저장')


---
## STEP 9 | 강건성 분석

> 전체·극단제외·USDT단독 3개 표본 × 3개 창. USDC 단독 과적합(R²≈1) → 각주용 별도 저장.


In [ ]:
EXTREME  = ['2023-03-10', '2022-11-11']
SVB_ONLY = ['2023-03-10']

ev_no_extreme = ev_all[
    ~ev_all['event_date'].dt.strftime('%Y-%m-%d').isin(EXTREME)].copy()

# ── USDC 단독·SVB제외 샘플 (데이터 구조 문제로 주 강건성 표 제외)
# 원인: USDC N=8에서 SVB(Q=1, CAR=-287bp)가 유일한 저품질 이벤트 →
#       quality_score가 완벽한 예측변수 → R²≈1, p<1e-20 (과적합)
# 처리: 별도 파일(robustness_usdc_note.csv)로 저장, 논문 각주 인용
ev_usdc_all   = ev_all[ev_all['coin']=='USDC'].copy()
ev_usdc_nosvb = ev_all[(ev_all['coin']=='USDC') &
                        ~ev_all['event_date'].dt.strftime('%Y-%m-%d').isin(SVB_ONLY)].copy()

print('='*70)
print('📌 Robustness Check (v5.1)')
print('   USDC 단독·SVB제외: 데이터 구조 문제로 주 표 제외, 각주 저장')
print('='*70)

# ── 주 강건성 샘플 (3개)
main_samples = [
    ('전체',           ev_all,          ['quality_score','negative','is_usdc']),
    ('극단이벤트제외', ev_no_extreme,   ['quality_score','negative','is_usdc']),
    ('USDT단독',       ev_all[ev_all['coin']=='USDT'], ['quality_score','negative']),
]

rob_rows = []
for name, df_sub, xcols in main_samples:
    n_total = len(df_sub)
    print(f'\n── {name}  (N={n_total}) ──')
    for car_col in ALL_WINS:
        sub = df_sub.dropna(subset=[car_col])
        n   = len(sub)
        if n < 4:
            print(f'  {car_col}: 관측치 부족({n})')
            continue
        X   = sm.add_constant(sub[xcols])
        lag = get_maxlags(n)
        dw0 = durbin_watson(sm.OLS(sub[car_col], X).fit().resid)
        if not (1.5 <= dw0 <= 2.5):
            m  = sm.OLS(sub[car_col], X).fit(
                    cov_type='HAC', cov_kwds={'maxlags': lag})
            se = f'HAC(lag={lag})'
        else:
            m  = sm.OLS(sub[car_col], X).fit(cov_type='HC3')
            se = 'HC3'
        coef    = m.params['quality_score']
        p_      = m.pvalues['quality_score']
        adj_r2  = m.rsquared_adj
        sig     = '***' if p_<0.01 else '**' if p_<0.05 else '*' if p_<0.1 else 'n.s.'
        r2note  = f'  ⚠ Adj-R²={adj_r2:.4f}(음수)' if adj_r2 < 0 else ''
        primary = ' ★' if car_col == PRIMARY_WIN else ''
        print(f'  {car_col}{primary}: {coef*10000:+.4f}bp  '
              f'p={p_:.4f} {sig}  R²={m.rsquared:.4f}  SE={se}{r2note}')
        rob_rows.append({
            'sample': name, 'window': car_col,
            'coef_bp': coef*10000, 'p': p_,
            'r2': m.rsquared, 'adj_r2': adj_r2,
            'N': n, 'se_type': se, 'lag': lag,
        })

pd.DataFrame(rob_rows).to_csv(
    f"{DIRS['results']}/robustness_check.csv", index=False)

# ── USDC 단독 별도 저장 (각주용)
note_rows = []
print('\n── [각주용] USDC 단독 (과적합 경고) ──')
for name, df_sub, xcols in [
    ('USDC단독',      ev_usdc_all,   ['quality_score','negative']),
    ('USDC(SVB제외)', ev_usdc_nosvb, ['quality_score','negative']),
]:
    n_total = len(df_sub)
    for car_col in ALL_WINS:
        sub = df_sub.dropna(subset=[car_col])
        n   = len(sub)
        if n < 4: continue
        X   = sm.add_constant(sub[xcols])
        lag = get_maxlags(n)
        dw0 = durbin_watson(sm.OLS(sub[car_col], X).fit().resid)
        if not (1.5 <= dw0 <= 2.5):
            m  = sm.OLS(sub[car_col], X).fit(
                    cov_type='HAC', cov_kwds={'maxlags': lag})
            se = f'HAC(lag={lag})'
        else:
            m  = sm.OLS(sub[car_col], X).fit(cov_type='HC3')
            se = 'HC3'
        coef   = m.params['quality_score']
        p_     = m.pvalues['quality_score']
        r2     = m.rsquared
        adj_r2 = m.rsquared_adj
        warning = ''
        if r2 > 0.9 and n < 12:
            warning = ' ❌ 과적합(이상치 주도, 결과 신뢰 불가)'
        sig = '***' if p_<0.01 else '**' if p_<0.05 else '*' if p_<0.1 else 'n.s.'
        print(f'  {name} {car_col}: {coef*10000:+.4f}bp  p={p_:.4e}  '
              f'R²={r2:.4f}  SE={se}{warning}')
        note_rows.append({
            'sample': name, 'window': car_col,
            'coef_bp': coef*10000, 'p': p_,
            'r2': r2, 'adj_r2': adj_r2,
            'N': n, 'se_type': se, 'lag': lag,
            'overfit_warning': bool(r2 > 0.9 and n < 12),
        })

pd.DataFrame(note_rows).to_csv(
    f"{DIRS['results']}/robustness_usdc_note.csv", index=False)
print('\n⚠ USDC 단독 결과는 robustness_usdc_note.csv에 저장 (논문 각주 인용)')
print('✅ 주 강건성 분석 완료')


---
## STEP 10 | 보완 분석

> 보완 1: 비정상 패그이탈 종속변수 H2. 보완 2: USDT 부정 이벤트 거래량 배율(도피처 증거).


In [ ]:
# 보완 1: 패그 이탈 기반 H2
print('📌 보완 분석 1: 종속변수 = 비정상 패그 이탈')
dev_rows = []
for _, ev in events_df.iterrows():
    ed, coin = ev['event_date'], ev['coin']
    dev_col  = 'dev_usdt' if coin == 'USDT' else 'dev_usdc'
    idx      = master.index[master['date'] == ed].tolist()
    if not idx:
        continue
    t0  = idx[0]
    pre = master.iloc[max(0,t0-60):t0-10].dropna(subset=[dev_col])
    win = master.iloc[max(0,t0-10):t0+11].dropna(subset=[dev_col])
    if len(pre) < 10:
        continue
    dev_rows.append({
        'event_date':    ed,
        'coin':          coin,
        'event_type':    ev['event_type'],
        'quality_score': ev['quality_score'],
        'abnormal_dev':  win[dev_col].mean() - pre[dev_col].mean(),
    })

dev_df = pd.DataFrame(dev_rows)
dev_df['negative'] = (dev_df['event_type'] == 'negative').astype(int)
dev_df['is_usdc']  = (dev_df['coin'] == 'USDC').astype(int)
dev_df.to_csv(f"{DIRS['results']}/abnormal_dev.csv", index=False)

print(dev_df.groupby(['coin','event_type'])['abnormal_dev']
      .agg(['mean','count']).round(8))

for nm, df_sub in [('전체',dev_df),
                   ('USDT',dev_df[dev_df['coin']=='USDT']),
                   ('USDC',dev_df[dev_df['coin']=='USDC'])]:
    if len(df_sub) < 5:
        continue
    xcols = (['quality_score','negative','is_usdc'] if nm=='전체'
             else ['quality_score','negative'])
    n  = len(df_sub)
    lag = get_maxlags(n)
    dw0 = durbin_watson(
        sm.OLS(df_sub['abnormal_dev'],
               sm.add_constant(df_sub[xcols])).fit().resid)
    if not (1.5 <= dw0 <= 2.5):
        m = sm.OLS(df_sub['abnormal_dev'],
                   sm.add_constant(df_sub[xcols])).fit(
                       cov_type='HAC', cov_kwds={'maxlags': lag})
    else:
        m = sm.OLS(df_sub['abnormal_dev'],
                   sm.add_constant(df_sub[xcols])).fit(cov_type='HC3')
    coef = m.params['quality_score']
    p_   = m.pvalues['quality_score']
    sig  = '***' if p_<0.01 else '**' if p_<0.05 else '*' if p_<0.1 else 'n.s.'
    print(f'  {nm:6s}(N={n}): quality coef={coef*10000:+.6f}  p={p_:.4f} {sig}')

# 보완 2: USDT Flight-to-Safety 거래량
print('\n📌 보완 분석 2: USDT Flight-to-Safety 거래량')
neg_usdt = events_df[(events_df['coin']=='USDT') &
                     (events_df['event_type']=='negative')]
for _, ev in neg_usdt.iterrows():
    idx = master.index[master['date'] == ev['event_date']].tolist()
    if not idx:
        continue
    t0    = idx[0]
    #  이벤트 창과 사전 기간 비중복 보장 (t0-11이 경계)
    w_vol = master.iloc[max(0,t0-5):t0+6]['v_usdt']   # 이벤트 창: τ∈[-5,+5]
    p_vol = master.iloc[max(0,t0-60):t0-11]['v_usdt'] # 사전 60일 (τ<-11)
    ratio = w_vol.mean() / p_vol.mean() if p_vol.mean() > 0 else np.nan
    car_v = ev_all[(ev_all['coin']=='USDT') &
                   (ev_all['event_date']==ev['event_date'])]['CAR_-3_3'].values
    car_s = f'{car_v[0]*10000:.2f}bp' if len(car_v) > 0 else 'N/A'
    print(f'  {str(ev["event_date"].date()):12s}  '
          f'{ev["description"][:40]:40s}  '
          f'거래량배율={ratio:.2f}x  CAR(-3,+3)={car_s}')

print('\n✅ 보완 분석 완료')


---
## STEP 11 | 데이터 검증

> 9개 항목 자동 검증: 결측/중복, 오염이벤트 미포함, USDT-BTC 상관, AR 편의, VIF, 정규성, DW/BP, Bootstrap 불일치.


In [ ]:
print('='*60)
print('📋 데이터 검증')
print('='*60)
val_log = []

# 1. 결측·중복
for sym, df in [('USDT',usdt),('USDC',usdc),('BTC',btc)]:
    dup  = df.date.duplicated().sum()
    null = df.price.isnull().sum()
    s    = '✅' if dup==0 and null==0 else '⚠'
    print(f'{sym}: 중복={dup}  결측={null}  {s}')
    val_log.append({'항목':f'{sym} 데이터','값':f'중복={dup}/결측={null}','판정':s})

# 2. 이벤트날짜 존재
mst  = set(master['date'].dt.date.astype(str))
miss = [str(r['event_date'].date()) for _,r in events_df.iterrows()
        if str(r['event_date'].date()) not in mst]
s    = '✅' if not miss else f'⚠ {miss}'
print(f'이벤트날짜 master 존재: {s}')
val_log.append({'항목':'이벤트날짜','값':len(miss),'판정':'✅' if not miss else '⚠'})

# 3. 제거된 오염이벤트 미포함 확인
used = set(ev_all['event_date'].dt.strftime('%Y-%m-%d'))
for exc in EXCLUDED_EVENTS:
    ok = exc['date'] not in used
    s  = '✅ 미포함' if ok else '❌ 포함됨!'
    print(f'오염이벤트 {exc["coin"]} {exc["date"]}: {s}')
    val_log.append({'항목':f'오염제거 {exc["coin"]} {exc["date"]}',
                    '값':exc['reason'],'판정':'✅' if ok else '❌'})

# 4. USDT-BTC 상관
corr = master[['r_usdt','r_btc']].dropna().corr().loc['r_usdt','r_btc']
s    = '✅' if abs(corr) < 0.15 else '⚠'
print(f'USDT-BTC 상관: r={corr:.6f}  {s}')
val_log.append({'항목':'USDT-BTC 상관','값':f'{corr:.6f}','판정':s})

# 5. AR 편의 검정
for cdf, coin in [(car_usdt,'USDT'),(car_usdc,'USDC')]:
    pre   = cdf[cdf['tau'] < 0]['AR'].dropna()
    _, p_ = ttest_1samp(pre, 0)
    s     = '✅' if p_ > 0.05 else '⚠'
    print(f'{coin} AR 편의(τ<0): p={p_:.4f}  {s}')
    val_log.append({'항목':f'{coin} AR 편의','값':f'p={p_:.4f}','판정':s})

# 6. VIF (절편 제외)
sub_v = ev_all.dropna(subset=[PRIMARY_WIN])
X_v   = sub_v[['quality_score','negative','is_usdc']]
X_va  = sm.add_constant(X_v)
vif   = [variance_inflation_factor(X_va.values, i)
         for i in range(1, X_va.shape[1])]   # 인덱스 1부터
vif_d = dict(zip(X_v.columns, [round(v,3) for v in vif]))
s     = '✅' if max(vif) < 5 else '⚠'
print(f'VIF(절편 제외): {vif_d}  max={max(vif):.3f}  {s}')
val_log.append({'항목':'VIF max(절편제외)','값':f'{max(vif):.3f}','판정':s})

# 7. 주 창 잔차 정규성
m_c   = sm.OLS(sub_v[PRIMARY_WIN], sm.add_constant(X_v)).fit()
_, sw = shapiro(m_c.resid)
s     = '✅' if sw > 0.05 else '⚠'
print(f'잔차 정규성({PRIMARY_WIN} Shapiro): p={sw:.4f}  {s}')
val_log.append({'항목':f'잔차 정규성({PRIMARY_WIN})','값':f'p={sw:.4f}','판정':s})

# 8. 시장모형 진단
# diag_all에서 직접 재계산 (STEP 4의 dw_w 변수 의존성 제거)
dw_n_chk = (diag_all['DW_판정'] == '⚠').sum()
bp_n_chk = (diag_all['BP_판정'] == '⚠').sum()
print(f'DW ⚠={dw_n_chk}건  BP ⚠={bp_n_chk}건  → HAC(동적 lag) 적용')
val_log.append({'항목':'DW 경고','값':f'{dw_n_chk}건','판정':'✅' if dw_n_chk==0 else '⚠'})
val_log.append({'항목':'BP 경고','값':f'{bp_n_chk}건','판정':'✅' if bp_n_chk==0 else '⚠'})

# 9. H1 Bootstrap vs t-test 불일치 항목 수
h1k  = pd.read_csv(f"{DIRS['results']}/H1_results.csv")
cnfl = h1k['conflict'].sum() if 'conflict' in h1k else 0
s    = '✅' if cnfl == 0 else f'⚠ {cnfl}건'
print(f'H1 CI·t-test 불일치: {s}')
val_log.append({'항목':'H1 Bootstrap-ttest 불일치','값':f'{cnfl}건',
                '판정':'✅' if cnfl==0 else '⚠'})

pd.DataFrame(val_log).to_csv(
    f"{DIRS['validation']}/validation_summary.csv",
    index=False, encoding='utf-8-sig')
pass_n = sum(v.startswith('✅') for v in pd.DataFrame(val_log)['판정'])
total  = len(val_log)
print(f'\n총 {total}항목: ✅{pass_n} / ⚠{total-pass_n}')


---
## STEP 12 | 시각화 (Figure 1~6)

> Figure 1~6 생성 (dpi=150). 한글 레이블, 이벤트창 명시, SVB FDIC 개입 주석선 포함.


In [ ]:
palette = {'positive':'#2ecc71','negative':'#e74c3c','neutral':'#95a5a6'}

# ── Figure 1
fig, axes = plt.subplots(2,1,figsize=(15,8),
                          gridspec_kw={'height_ratios':[2,1]})
ax = axes[0]
ax.plot(master['date'],master['p_usdt'],lw=1.2,color='#1f77b4',label='USDT')
ax.plot(master['date'],master['p_usdc'],lw=1.2,color='#ff7f0e',label='USDC')
ax.axhline(1.0,color='black',lw=0.8,ls='--',alpha=0.4)
for _,ev in events_df.iterrows():
    ax.axvline(ev['event_date'],color=palette[ev['event_type']],lw=0.6,alpha=0.3)
ax.legend(handles=[
    plt.Line2D([0],[0],color='#1f77b4',lw=2,label='USDT'),
    plt.Line2D([0],[0],color='#ff7f0e',lw=2,label='USDC'),
    Patch(facecolor='#2ecc71',alpha=0.5,label='긍정'),
    Patch(facecolor='#95a5a6',alpha=0.5,label='중립'),
    Patch(facecolor='#e74c3c',alpha=0.5,label='부정'),
],fontsize=9)
ax.set_title('Figure 1. USDT·USDC 가격 시계열 & 이벤트 분포',fontsize=12,fontweight='bold')  # 
ax.set_ylabel('가격(USD)'); ax.grid(alpha=0.3); ax.set_ylim(0.95,1.04)
ax2 = axes[1]
for coin,color in [('USDT','#1f77b4'),('USDC','#ff7f0e')]:
    s = events_df[events_df['coin']==coin].sort_values('event_date')
    ax2.plot(s['event_date'],s['quality_score'],'o-',color=color,lw=1.5,ms=6,label=coin)
ax2.set_title('공시 품질 지수 추이',fontsize=10)
ax2.set_ylabel('공시 품질 지수(점)',fontsize=9)  # 
ax2.set_ylim(0,6); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DIRS['figures']}/Fig1_price_events.png",dpi=150,bbox_inches='tight')
plt.show()

# ── Figure 2: 평균 CAR 추이
fig,axes = plt.subplots(1,2,figsize=(14,5))
for ax,(s,coin,color) in zip(axes,[
        (sum_usdt,'USDT','#1f77b4'),(sum_usdc,'USDC','#ff7f0e')]):
    ax.fill_between(s['tau'],(s['mean_CAR']-1.96*s['se'])*10000,
                    (s['mean_CAR']+1.96*s['se'])*10000,alpha=0.12,color=color)
    ax.plot(s['tau'],s['mean_CAR']*10000,color=color,lw=2,marker='o',ms=3.5)
    ax.axhline(0,color='black',lw=0.8,ls='--',alpha=0.5)
    ax.axvline(0,color='red',lw=1.2,ls=':',label='공시일')
    # USDC 패널에 SVB 포함 안내 추가
    svb_note = ' (SVB 이벤트 포함)' if coin == 'USDC' else ''
    ax.set_title(f'Figure 2. {coin} 평균 CAR(bp)  N={s["n"].mode()[0]}{svb_note}',
                 fontsize=12,fontweight='bold')
    ax.set_xlabel('τ'); ax.set_ylabel('CAR(bp)'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DIRS['figures']}/Fig2_car_timeline.png",dpi=150,bbox_inches='tight')
plt.show()

# ── Figure 3: 유형별 박스플롯
fig,axes = plt.subplots(1,2,figsize=(12,5))
for ax,(ev_p,coin) in zip(axes,[(ev_usdt,'USDT'),(ev_usdc,'USDC')]):
    data  = ev_p.dropna(subset=['CAR_-3_3'])
    order = [t for t in ['positive','neutral','negative']
             if t in data['event_type'].values]
    bp = ax.boxplot(
        [data[data['event_type']==t]['CAR_-3_3'].values*10000 for t in order],
        labels=[{'positive':'긍정','neutral':'중립','negative':'부정'}[t] for t in order],
        patch_artist=True,widths=0.5,medianprops=dict(color='black',lw=2))
    for patch,t in zip(bp['boxes'],order):
        patch.set(facecolor=palette[t],alpha=0.75)
    ax.axhline(0,color='black',lw=0.8,ls='--',alpha=0.5)
    ax.set_title(f'Figure 3. {coin} 이벤트 유형별 CAR(-3,+3)  [CAR(-3,+3) 기준]',fontsize=12,fontweight='bold')  # 이벤트 창 명시
    ax.set_ylabel('CAR(bp)'); ax.grid(alpha=0.3,axis='y')
plt.tight_layout()
plt.savefig(f"{DIRS['figures']}/Fig3_car_boxplot.png",dpi=150,bbox_inches='tight')
plt.show()

# ── Figure 4: 공시품질 × CAR 산점도 (주 창)
fig,axes = plt.subplots(1,2,figsize=(12,5))
for ax,(ev_p,coin,color) in zip(axes,[
        (ev_usdt,'USDT','#1f77b4'),(ev_usdc,'USDC','#ff7f0e')]):
    data = ev_p.dropna(subset=[PRIMARY_WIN])
    ax.scatter(data['quality_score'],data[PRIMARY_WIN]*10000,
               c=data['event_type'].map(palette),
               s=90,alpha=0.85,edgecolors='white',lw=0.8,zorder=3)
    z  = np.polyfit(data['quality_score'],data[PRIMARY_WIN]*10000,1)
    xs = np.linspace(0.8,5.2,50)
    ax.plot(xs,np.poly1d(z)(xs),color=color,lw=2,ls='--',
            label=f'단순회귀 기울기={z[0]:.3f}bp/점')
    ax.axhline(0,color='black',lw=0.8,alpha=0.5)
    ax.set_title(f'Figure 4. {coin} 공시품질 × {PRIMARY_WIN}',
                 fontsize=12,fontweight='bold')
    ax.set_xlabel('공시 품질 지수(점)'); ax.set_ylabel('CAR(bp)')  #  x축 한글화
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DIRS['figures']}/Fig4_quality_scatter.png",dpi=150,bbox_inches='tight')
plt.show()

# ── Figure 5: SVB 케이스스터디
svb = pd.Timestamp('2023-03-10')
w5  = master[(master['date']>=svb-pd.Timedelta('30d')) &
             (master['date']<=svb+pd.Timedelta('30d'))]
fig,(a1,a2) = plt.subplots(2,1,figsize=(14,8),sharex=False)
a1.plot(w5['date'],w5['p_usdc'],color='#ff7f0e',lw=2)
a1.axhline(1.0,color='black',lw=0.8,ls='--')
a1.axvline(svb,color='red',lw=1.5,ls=':',label='SVB 파산(3/10)')
a1.axvline(svb+pd.Timedelta('3d'),color='gray',lw=1.5,ls='--',
           label='패그회복(3/13) — 오염이벤트 제거')
a1.fill_between(w5['date'],w5['p_usdc'],1.0,
                where=w5['p_usdc']<1.0,alpha=0.2,color='red')
a1.set_title('Figure 5. SVB 케이스스터디',fontsize=12,fontweight='bold')
a1.set_ylabel('USDC 가격'); a1.legend(fontsize=9); a1.grid(alpha=0.3)
svb_car = car_usdc[car_usdc['event_date']==svb].sort_values('tau')
if len(svb_car)>0:
    a2.bar(svb_car['tau'],svb_car['AR']*10000,
           color=np.where(svb_car['AR']<0,'#e74c3c','#2ecc71'),alpha=0.7)
    a2.plot(svb_car['tau'],svb_car['CAR']*10000,
            color='#ff7f0e',lw=2.5,marker='o',ms=4,label='CAR')
    a2.axhline(0,color='black',lw=0.8)
    a2.axvline(0,color='red',lw=1.2,ls=':')
    #  τ=+2 반등 주석선 — FDIC 개입(3/13) 반등, 오염 이벤트로 별도 제외
    a2.axvline(2,color='gray',lw=1.2,ls='--',alpha=0.7,
               label='τ=+2: FDIC 개입 반등 (3/13 오염이벤트 제외)')
    a2.set_xlabel('τ'); a2.set_ylabel('비정상수익률(bp)')
    a2.legend(fontsize=8,loc='upper right'); a2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DIRS['figures']}/Fig5_svb_case.png",dpi=150,bbox_inches='tight')
plt.show()

# ── Figure 6: 강건성
rob_df = pd.read_csv(f"{DIRS['results']}/robustness_check.csv")
#  robustness_check.csv에는 주 3개 샘플만 존재 (USDC 샘플 제거)
s_ord  = ['전체','극단이벤트제외','USDT단독']
cols6  = ['#1f77b4','#2ecc71','#ff7f0e']
x = np.arange(3); w = 0.15
fig,ax = plt.subplots(figsize=(12,5))
for i,(samp,color) in enumerate(zip(s_ord,cols6)):
    sub  = rob_df[rob_df['sample']==samp].set_index('window')
    vals = [sub.loc[ww,'coef_bp'] if ww in sub.index else 0 for ww in ALL_WINS]
    ps   = [sub.loc[ww,'p']       if ww in sub.index else 1 for ww in ALL_WINS]
    ax.bar(x+i*w,vals,w,label=samp,color=color,alpha=0.8,edgecolor='white')
    for j,(v,p) in enumerate(zip(vals,ps)):
        if p<0.05:
            ax.text(x[j]+i*w,v+(0.3 if v>=0 else -0.8),'*',ha='center',fontsize=12)
ax.axhline(0,color='black',lw=0.8)
ax.set_xticks(x+w*2)
ax.set_xticklabels(['CAR(-1,+1)\n(표본별 불안정)','CAR(-3,+3)',f'CAR(-5,+5) ★주창'])  #  불안정성 표기
ax.set_title('Figure 6. Robustness: quality_score 계수(bp)  *=p<0.05',
             fontsize=12,fontweight='bold')
ax.set_ylabel('계수(bp)'); ax.legend(fontsize=9,ncol=3)
ax.grid(alpha=0.3,axis='y')
plt.tight_layout()
plt.savefig(f"{DIRS['figures']}/Fig6_robustness.png",dpi=150,bbox_inches='tight')
plt.show()

print('✅ Figure 1~6 저장 완료')


---
## STEP 13 | 테이블 & 최종 요약

> Table 2 (이벤트별 CAR) 생성. H1·H2·H3 결과 집약 출력. 


In [ ]:
# Table 2
t2 = ev_all[['coin','event_date','event_type','quality_score',
              'CAR_-1_1','CAR_-3_3','CAR_-5_5']].copy()
t2['event_date'] = t2['event_date'].dt.strftime('%Y-%m-%d')
for c in ['CAR_-1_1','CAR_-3_3','CAR_-5_5']:
    t2[c] = (t2[c] * 10000).round(3)
t2.columns = ['코인','공시일','유형','품질지수',
               'CAR(-1,+1)bp','CAR(-3,+3)bp','CAR(-5,+5)bp']
t2.to_csv(f"{DIRS['tables']}/Table2_event_CAR.csv",
          index=False, encoding='utf-8-sig')
print(f'Table 2: {len(t2)}건  ✅')

print('\n'+'='*70)
print('📋 최종 분석 결과 요약 (v5)')
print('='*70)
print(f'  기간: {master.date.min().date()} ~ {master.date.max().date()} ({len(master)}일)')
print(f'  이벤트: USDT {len(ev_usdt)}건 / USDC {len(ev_usdc)}건 = {len(ev_all)}건')
print(f'  주 창: {PRIMARY_WIN}  (잔차 정규성 통과)')
print()

h1k = pd.read_csv(f"{DIRS['results']}/H1_results.csv")
print('[H1]:')
for _,r in h1k[h1k['window']==PRIMARY_WIN].iterrows():
    sig = ('***' if r['p']<0.01 else '**' if r['p']<0.05
           else '*' if r['p']<0.1 else 'n.s.')
    cnfl = '  ⚠ Boot·t 불일치' if r.get('conflict',False) else ''
    print(f'  {r["coin"]:4s}  mean={r["mean_bp"]:+8.3f}bp  '
          f't={r["t"]:6.3f}  p={r["p"]:.4f} {sig}  '
          f'Boot95%=[{r["boot_lo_bp"]:+.3f},{r["boot_hi_bp"]:+.3f}]{cnfl}')

h2k = pd.read_csv(f"{DIRS['results']}/H2_regression.csv")
pw  = h2k[(h2k['var']=='quality_score')&(h2k['window']==PRIMARY_WIN)].iloc[0]
dg  = pd.read_csv(f"{DIRS['results']}/H2_residual_diagnostics.csv")
dw_row = dg[dg['window']==PRIMARY_WIN].iloc[0]
adj = dw_row['AdjR2']
adj_note = '  ⚠ 음수' if adj < 0 else ''
print(f'\n[H2] {PRIMARY_WIN}: '
      f'coef={pw.coef_bp:.4f}bp  p={pw.p:.4f}  '
      f'R²={dw_row["R2"]:.4f}  Adj-R²={adj:.4f}{adj_note}  '
      f'VIF_max={dw_row["VIF_max"]:.3f}  SE={pw.se_type}(lag={int(pw.lag)})')

h3k = pd.read_csv(f"{DIRS['results']}/H3_results.csv")
print(f'\n[H3] {PRIMARY_WIN}:')
for _,r in h3k[h3k['window']==PRIMARY_WIN].iterrows():
    rn = r['ratio_note'] if pd.notna(r.get('ratio_note','')) else ''
    rv = f'{r["ratio"]:.2f}x ({rn})' if pd.notna(r['ratio']) else rn
    if r['test_feasible']:
        sig = ('***' if r['p_ttest']<0.01 else '**' if r['p_ttest']<0.05
               else '*' if r['p_ttest']<0.1 else 'n.s.')
        print(f'  {r["coin"]:4s}  긍정={r["pos_mean_bp"]:+8.3f}bp  '
              f'부정={r["neg_mean_bp"]:+8.3f}bp  비율={rv}  '
              f'p={r["p_ttest"]:.4f}{sig}  [{r["direction"]}]')
    else:
        print(f'  {r["coin"]:4s}  검정불가(N_neg={r["n_neg"]})  '
              f'부정={r["neg_mean_bp"]:+8.3f}bp  비율={rv}  [{r["direction"]}]')

print()

